# VnStock Data
Notebook này được xây dựng để thử nghiệm các chức năng cơ bản dựa trên mục tiêu dự án với thư viện Vnstock - Sàn chứng khoán Việt Nam.
- Khả năng truy cập Real-time
- Vẽ biểu đổ demo Real-time
- Lấy dữ liệu từ quá khứ dựa trên một vài mã cổ phiếu

## 1. Chuẩn bị

In [16]:
from vnstock import *
from pathlib import Path

In [17]:
try:
    api_path = Path("api.txt")
    api_key = api_path.read_text(encoding="utf-8").strip()
    api_key = api_key[10:20]
    print("Đã đọc API key:", api_key + "...")
except Exception as e:
    print("Copy file api.demo.txt thành api.txt và điền API key vào đó nhé!")
    api_key = None

Đã đọc API key: vnstock_73...


In [39]:
if api_key:
    register_user(api_key)
else:
    print("Không có API key, giới hạn 20 lần gọi API mỗi phút")

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***k_73
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)


## 2. Kiểm tra kết nối

In [38]:
from vnstock import Vnstock

stock = Vnstock().stock(symbol='VCB', source='VCI')

df = stock.quote.intraday()
df.tail()

,time,price,volume,match_type,id
95,2026-04-14 14:29:40+07:00,59.3,700,Buy,463626797
96,2026-04-14 14:29:51+07:00,59.3,100,Buy,463627444
97,2026-04-14 14:29:57+07:00,59.3,100,Buy,463628028
98,2026-04-14 14:29:58+07:00,59.3,100,Buy,463628060
99,2026-04-14 14:45:00+07:00,59.3,160700,ATC,463629612


## 3. Đặc điểm dữ liệu
### 3.1 Mã chứng khoán ( KBS )

#### Danh sách tất cả mã CK niêm yết (đơn giản)

In [40]:
listing = Listing(source='KBS')
listing.all_symbols()

,symbol,organ_name
0,DPP,CTCP Dược Đồng Nai
1,SDA,CTCP Simco Sông Đà
2,CLH,CTCP Xi măng La Hiên VVMI
3,DBT,CTCP Dược phẩm Bến Tre
4,DND,CTCP Đầu tư Xây dựng và Vật liệu Đồng Nai
...,...,...
1535,ABT,CTCP Xuất nhập khẩu Thủy sản Bến Tre
1536,DHG,CTCP Dược Hậu Giang
1537,PEG,Tổng Công ty Thương mại Kỹ thuật và Đầu tư - CTCP
1538,POB,CTCP Xăng dầu Dầu khí Hưng Yên


#### Liệt kê mã CP theo sàn (đầy đủ)

In [41]:
# Dữ liệu bao gồm tất cả các mã không kể đang niêm yết hay không
listing.symbols_by_exchange()

,symbol,organ_name,en_organ_name,exchange,type,id
0,MGC,CTCP Địa chất mỏ - TKV,Vinacomin - Mining Geology Joint Stock Company,UPCOM,stock,1
1,GVT,CTCP Giấy Việt Trì,Viet Tri Paper Joint Stock Company,UPCOM,stock,1
2,GEG,CTCP Điện Gia Lai,Gia Lai Electricity Joint Stock Company,HOSE,stock,1
3,SWC,Tổng Công ty cổ phần Đường sông Miền Nam,Southern Waterborne Transport Corporation,UPCOM,stock,1
4,SLD,CTCP Địa ốc Sacom,Sacom Land Corporation,UPCOM,stock,1
...,...,...,...,...,...,...
3249,TDK12102,NaN,NaN,NaN,corpbond,7
3250,PSI12501,NaN,NaN,NaN,corpbond,7
3251,MBB12110,NaN,NaN,NaN,corpbond,7
3252,HNV32202,NaN,NaN,NaN,corpbond,7


#### Liệt kê mã CP theo ngành icb

In [42]:
listing.symbols_by_industries()

,symbol,industry_code,industry_name
0,AAV,3,Bất động sản
1,AGG,3,Bất động sản
2,API,3,Bất động sản
3,BAX,3,Bất động sản
4,BCE,3,Bất động sản
...,...,...,...
693,SSM,26,SX Phụ trợ
694,SVT,26,SX Phụ trợ
695,TLD,26,SX Phụ trợ
696,TLG,26,SX Phụ trợ


In [43]:
# Liệt kê tất cả mã chứng khoán theo nhóm phân loại. Ví dụ HOSE, VN30, VNMidCap, VNSmallCap, VNAllShare, VN100, ETF, HNX, HNX30, HNXCon, HNXFin, HNXLCap, HNXMSCap, HNXMan, UPCOM, FU_INDEX (mã chỉ số hợp đồng tương lai), CW (chứng quyền)
listing.symbols_by_group('VN30')

0     ACB
1     BID
2     CTG
3     DGC
4     FPT
5     GAS
6     GVR
7     HDB
8     HPG
9     LPB
10    MBB
11    MSN
12    MWG
13    PLX
14    SAB
15    SHB
16    SSB
17    SSI
18    STB
19    TCB
20    TPB
21    VCB
22    VHM
23    VIB
24    VIC
25    VJC
26    VNM
27    VPB
28    VPL
29    VRE
Name: symbol, dtype: str

### 3.2 Mã chứng khoán (VCI)

## 4. Khả năng lấy dữ liệu lịch sử 30 ngày trước

### 4.1 Một mã cổ phiếu